## env: GPU

In [20]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn as nn
import torch.optim as optim

import time

import shutil

import torchvision.utils
from torch.utils.data import DataLoader, Subset
from torchvision import models
import torchvision.datasets as dsets
import torchvision.transforms as transforms

import torchattacks
from torchattacks import PGD, FGSM
from torchsummary import summary
from sklearn.model_selection import train_test_split

In [21]:
batch_size = 20

trainset = torchvision.datasets.ImageFolder(
    root='./data/GTSRB/Final_Training/Images',
    transform=transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
    ])
)

testset = torchvision.datasets.ImageFolder(
    root='./data/GTSRB/test',
    transform=transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
    ])
)

# indices = list(range(len(full_trainset)))
# labels = full_trainset.targets

# train_indices, val_indices = train_test_split(
#     indices,
#     test_size=0.2,
#     stratify=labels,
#     random_state=42
# )

# trainset = Subset(full_trainset, train_indices)
# valset = Subset(full_trainset, val_indices)

train_loader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4
)

# val_loader = DataLoader(
#     valset,
#     batch_size=batch_size,
#     shuffle=False,
#     num_workers=4
# )

test_loader = DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

## model LNL

In [22]:
from LNL import LNL_Ti as small
model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
model = model.cuda()

## Train Locality-iN-Locality

In [23]:
num_epochs = 40

In [24]:
checkpoint_dir = '../checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

Validation function

In [25]:
def validate(model, val_loader, criterion):
    model.eval()

    val_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.cuda()
            labels = labels.cuda()

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    avg_loss = val_loss / len(val_loader)
    accuracy = 100 * correct / total

    return avg_loss, accuracy

In [ ]:
def evaluate_model(model, test_loader, device=None):
    
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    return accuracy

In [31]:
def train_model(start_epoch, end_epoch):
    best_acc = 0

    loss = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
    # scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs,eta_min=0)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.001,
        steps_per_epoch=len(train_loader),
        epochs=end_epoch - start_epoch
    )

    # Nếu tiếp tục train từ checkpoint
    if start_epoch != 0:
        checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{start_epoch}.pth")
        checkpoint = torch.load(checkpoint_path, map_location="cuda")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        best_acc = checkpoint.get("best_acc", 0)
        print(f"Loaded checkpoint from epoch {start_epoch}")

    # Train từ start_epoch đến end_epoch
    for epoch in range(start_epoch, end_epoch):
        model.train()

        train_loss, correct, total = 0.0, 0, 0

        for i, (batch_images, batch_labels) in enumerate(train_loader):
            X = batch_images.cuda()
            Y = batch_labels.cuda()

            pre = model(X)
            cost = loss(pre, Y)
            optimizer.zero_grad()
            cost.backward()
            optimizer.step()

            train_loss += cost.item()
            _, predicted = pre.max(1)
            total += Y.size(0)
            correct += predicted.eq(Y).sum().item()

            # Cập nhật learning rate
            # scheduler.step()

        train_loss /= len(train_loader)
        train_acc = 100.0 * correct / total

        lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch [{epoch+1}/{num_epochs}]: LR: {lr:.6f}\n"
            f"\tTrain Loss: {train_loss:.8f} | Train Acc: {train_acc:.2f}%"
        )
        checkpoint_path = os.path.join(
            checkpoint_dir,
            f"checkpoint_epoch_{epoch + 1}.pth"
        )
        if train_acc >= best_acc:
            best_acc = train_acc

            test_acc = evaluate_model(model, test_loader)
            print(f"\tTest Acc: {test_acc:.2f}% {checkpoint_path}")
            

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_acc": best_acc,
        }, checkpoint_path)

In [32]:
start_epoch = 0
end_epoch = 50
train_model(start_epoch, end_epoch)

Epoch [1/40]: LR: 0.000040
	Train Loss: 0.69014899 | Train Acc: 99.95%
	Test Acc: 98.61% ../checkpoints\checkpoint_epoch_1.pth
Epoch [2/40]: LR: 0.000040
	Train Loss: 0.68774487 | Train Acc: 99.99%
	Test Acc: 98.73% ../checkpoints\checkpoint_epoch_2.pth
Epoch [3/40]: LR: 0.000040
	Train Loss: 0.68699881 | Train Acc: 99.99%
	Test Acc: 98.73% ../checkpoints\checkpoint_epoch_3.pth
Epoch [4/40]: LR: 0.000040
	Train Loss: 0.68645106 | Train Acc: 100.00%
	Test Acc: 98.85% ../checkpoints\checkpoint_epoch_4.pth
Epoch [5/40]: LR: 0.000040
	Train Loss: 0.68619601 | Train Acc: 100.00%
	Test Acc: 98.85% ../checkpoints\checkpoint_epoch_5.pth
Epoch [6/40]: LR: 0.000040
	Train Loss: 0.68594461 | Train Acc: 100.00%
	Test Acc: 98.84% ../checkpoints\checkpoint_epoch_6.pth
Epoch [7/40]: LR: 0.000040
	Train Loss: 0.68574933 | Train Acc: 100.00%
	Test Acc: 98.95% ../checkpoints\checkpoint_epoch_7.pth
Epoch [8/40]: LR: 0.000040
	Train Loss: 0.68561134 | Train Acc: 100.00%
	Test Acc: 98.96% ../checkpoints\ch

KeyboardInterrupt: 

## Test

In [28]:
model.eval()
correct = 0
total = 0

for images, labels in test_loader:
    images = images.cuda()
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()

print('Standard accuracy: %.2f %%' % (100 * float(correct) / total))

Standard accuracy: 97.93 %


Test a checkpoint

In [ ]:
checkpoint_path = "../checkpoints/99_33.pth"
torch.save({
                "model_state_dict": model.state_dict(),
            }, checkpoint_path)

test_model = small(pretrained=False)
test_model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
checkpoint = torch.load(checkpoint_path, map_location="cuda")
test_model.load_state_dict(checkpoint["model_state_dict"])
test_model = test_model.cuda()

test_model.eval()

correct = 0
total = 0

for images, labels in test_loader:
    images = images.cuda()
    outputs = test_model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
    
print('Standard accuracy: %.2f %%' % (100 * float(correct) / total))

Test all checkpoints in checkpoints dir

In [10]:
def evaluate_model(model, test_loader, device=None):
    """
    Evaluate model on test set
    """
    
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    return accuracy

In [ ]:
import glob
from LNL import LNL_Ti as small

def test_checkpoints(checkpoint_dir, test_loader, num_classes=43, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Lấy tất cả file .pth
    checkpoint_paths = glob.glob(os.path.join(checkpoint_dir, "*.pth"))

    # Sắp xếp theo tên file
    checkpoint_paths = sorted(checkpoint_paths)

    results = {}

    for checkpoint_path in checkpoint_paths:
        checkpoint_name = os.path.basename(checkpoint_path)
        print(f"\nTesting: {checkpoint_name}")

        # Tạo model
        test_model = small(pretrained=False)

        # Thay classification head
        test_model.head = nn.Linear(in_features=192, out_features=num_classes, bias=True)

        # Đưa model lên device
        test_model = test_model.to(device)

        # Load checkpoint
        checkpoint = torch.load(checkpoint_path, map_location=device)

        test_model.load_state_dict(checkpoint["model_state_dict"])

        # Evaluate
        accuracy = evaluate_model(test_model, test_loader, device)

        results[checkpoint_name] = accuracy

        print(f"Accuracy: {accuracy:.2f}%")

    return results


checkpoint_dir = "../checkpoints"
results = test_checkpoints(checkpoint_dir=checkpoint_dir, test_loader=test_loader, num_classes=43)
print(results)

## FGSM attack

In [ ]:
model.eval()

correct = 0
total = 0

atk = FGSM(model, eps=0.01)

for images, labels in test_loader:
    
    images = atk(images, labels).cuda()
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
    
print('Robust accuracy: %.2f %%' % (100 * float(correct) / total))

## PGD attack

In [ ]:
model.eval()

correct = 0
total = 0

atk = PGD(model, eps=0.01, alpha=2/255, steps=5, random_start=False)

for images, labels in test_loader:
    
    images = atk(images, labels).cuda()
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
    
print('Robust accuracy: %.2f %%' % (100 * float(correct) / total))

## train LNL-MoEx

In [ ]:
from LNL_MoEx import LNL_MoEx_Ti as small
model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
model = model.cuda()

In [ ]:
import time
# time.clock_gettime()

In [ ]:
num_epochs = 20
moex_lam = .9
moex_prob = .7

In [ ]:
def train_model_TNT_MoEx(start_epoch, end_epoch):
    loss = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs,eta_min=0)

    # Nếu tiếp tục train từ checkpoint
    if start_epoch != 0:
        checkpoint_path = os.path.join(checkpoint_path, f"checkpoint_epoch_{start_epoch}.pth")
        checkpoint = torch.load(checkpoint_path, map_location="cuda")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        print(f"Loaded checkpoint from epoch {start_epoch}")

    # Train từ start_epoch đến end_epoch
    for epoch in range(start_epoch, end_epoch):
        total_batch = len(train_loader)
        model.train()

        for i, (input, target) in enumerate(train_loader):
            input = input.cuda()
            target = target.cuda()

            prob = torch.rand(1).item()

            if prob < moex_prob:
                swap_index = torch.randperm(input.size(0), device=input.device)
                with torch.no_grad():
                    target_a = target
                    target_b = target[swap_index]
                output = model(input, swap_index=swap_index, moex_norm='pono', moex_epsilon=1e-5,
                                moex_layer='stem', moex_positive_only=False)
                lam = moex_lam
                cost = loss(output, target_a) * lam + loss(output, target_b) * (1. - lam)
            else:
                # compute output
                output = model(input)
                # if args.prof >= 0: torch.cuda.nvtx.range_pop()
                cost = loss(output, target)

            optimizer.zero_grad()
            cost.backward()
            optimizer.step()

            if (i + 1) % 200 == 0:
                print(
                    'Epoch [%d/%d], Iter [%d/%d], Loss: %.6f'
                    % (epoch + 1, num_epochs, i + 1, total_batch, cost.item())
                )

        # Cập nhật learning rate
        scheduler.step()

        # Lưu checkpoint sau mỗi 10 epoch
        if (epoch + 1) % 10 == 0:
            checkpoint_path = os.path.join(
                checkpoint_dir,
                f"checkpoint_epoch_{epoch + 1}.pth"
            )

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
            }, checkpoint_path)

            print(f"Checkpoint saved to: {checkpoint_path}\n")

In [ ]:
start_epoch = 0
end_epoch = 20
train_model_TNT_MoEx(start_epoch, end_epoch)

## Number of Parameters

In [ ]:
# pip install ptflops

In [ ]:
# pip install --upgrade git+https://github.com/sovrasov/flops-counter.pytorch.git

In [ ]:
# import torch
# from ptflops import get_model_complexity_info

# with torch.cuda.device(0):
#   net = model
#   macs, params = get_model_complexity_info(net, (3, 224, 224), as_strings=True,
#                                            print_per_layer_stat=True, verbose=True)
#   print('{:<30}  {:<8}'.format('Computational complexity: ', macs))
#   print('{:<30}  {:<8}'.format('Number of parameters: ', params))
